# Lat/Long Finder (Google Colab)

Upload a spreadsheet (`.csv`, `.xls`, `.xlsx`) with a **Full Address** column (at minimum: Number + Street + City/Province) and this notebook will geocode it using the free Esri ArcGIS `findAddressCandidates` endpoint, then let you download the result with `latitude`, `longitude`, `geocode_score`, `matched_address`, and `address_type` columns added.

In [ ]:
!pip install -q pandas requests tqdm openpyxl

In [ ]:
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm
from google.colab import files

GEOCODE_URL = "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates"

RESULT_COLUMNS = [
    "latitude",
    "longitude",
    "geocode_score",
    "matched_address",
    "address_type",
]


def geocode_arcgis(full_address, country_code="CAN", timeout=10):
    params = {
        "f": "json",
        "singleLine": full_address,
        "outFields": "Match_addr,Addr_type,Score",
        "maxLocations": 1,
    }
    if country_code:
        params["countryCode"] = country_code

    try:
        response = requests.get(GEOCODE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        data = response.json()

        candidates = data.get("candidates") or []
        if candidates:
            best = candidates[0]
            return {
                "latitude": best["location"]["y"],
                "longitude": best["location"]["x"],
                "geocode_score": best["attributes"]["Score"],
                "matched_address": best["attributes"]["Match_addr"],
                "address_type": best["attributes"]["Addr_type"],
            }
    except Exception as e:
        print(f"Error geocoding '{full_address}': {e}")

    return {
        "latitude": None,
        "longitude": None,
        "geocode_score": None,
        "matched_address": None,
        "address_type": None,
    }

In [ ]:
# @title Upload database with a "Full Address" column (mandatory: at least Number + Street + Province)

COUNTRY_CODE = "CAN"  # @param {type:"string"}
REQUEST_DELAY_SECONDS = 0.05  # @param {type:"number"}

print("Upload your Excel file (.xls, .xlsx, .csv)")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"\nFile uploaded: {file_name}")

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print(f"\nTotal records: {len(df)}")
print(f"Columns found: {df.columns.tolist()}\n")

address_col = None
for col in df.columns:
    if "full" in col.lower() and "address" in col.lower():
        address_col = col
        break

if not address_col:
    print("Available columns:")
    for i, col in enumerate(df.columns):
        print(f"{i}: {col}")

    col_index = int(input("\nEnter the column number for Full Address: "))
    address_col = df.columns[col_index]

print(f"Using column: '{address_col}' for geocoding\n")

# Drop any stale geocode result columns from a previous run so the new
# results aren't concatenated alongside duplicate column names (which
# turns df_final['latitude'] into a multi-column DataFrame downstream and
# breaks the summary f-strings below).
df = df.drop(columns=[c for c in RESULT_COLUMNS if c in df.columns])

In [ ]:
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Geocoding"):
    full_address = str(row[address_col])
    results.append(geocode_arcgis(full_address, country_code=COUNTRY_CODE or None))
    time.sleep(REQUEST_DELAY_SECONDS)

df_geo = pd.DataFrame(results, columns=RESULT_COLUMNS)
df_final = pd.concat([df.reset_index(drop=True), df_geo], axis=1)

success_count = int(df_final["latitude"].notna().sum())
unique_coords = df_final[["latitude", "longitude"]].drop_duplicates()

print("\nGeocoding complete!")
print(f"Successful: {success_count}/{len(df_final)} ({success_count / len(df_final) * 100:.1f}%)")
print(f"Unique locations: {len(unique_coords)}")

high_quality = df_final[df_final["geocode_score"] >= 90]
print(f"High quality matches (score >= 90): {len(high_quality)}")

In [ ]:
output_name = file_name.rsplit(".", 1)[0] + "_geocoded"
df_final.to_csv(f"{output_name}.csv", index=False)
df_final.to_excel(f"{output_name}.xlsx", index=False)

print("\nFiles saved:")
print(f"- {output_name}.csv")
print(f"- {output_name}.xlsx")

print("\nSample of geocoded data:")
print(df_final[[address_col, "latitude", "longitude", "geocode_score"]].head(10))

print("\nDownloading files...")
files.download(f"{output_name}.csv")
files.download(f"{output_name}.xlsx")